# autoresearch â€” A4000 Cloud Notebook Setup

Karpathy's autoresearch: AI agent autonomously runs 5-min training experiments, keeps improvements, discards failures, and loops indefinitely.

**Cloud kernel: RTX A4000 (16 GB VRAM), PyTorch 2.1.1+cu121, Python 3.11.** Uses `pip` (no uv on this kernel).

**Steps:** 1. GPU check â†’ 2. Install deps â†’ 3. A4000 patches â†’ 4. Research libraries â†’ 5. Prepare data â†’ 6. Baseline run â†’ 7. Agent loop

## 1. GPU Check

In [5]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    cap = torch.cuda.get_device_capability()
    arch = "Hopper" if cap == (9,0) else "Ampere/Ada" if cap[0] == 8 else "Other"
    print(f"Cap:     {cap} ({arch})")

NVIDIA RTX A4000, 16376 MiB, 8.6

PyTorch: 2.1.1+cu121
CUDA:    True
GPU:     NVIDIA RTX A4000
VRAM:    15.7 GB
Cap:     (8, 6) (Ampere/Ada)


## 2. Install Core Dependencies

Cloud kernel uses `pip` directly. Torch 2.1.1+cu121 is already present â€” skip reinstalling it unless you need a specific version.

In [6]:
import sys, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args, "-q"])
    return r.returncode == 0

# Core project deps (skip torch â€” already installed on cloud kernel)
core = [
    "kernels",       # FA3 / GPU kernel loader
    "matplotlib",    # plotting
    "numpy",
    "pandas",
    "pyarrow",       # parquet reading
    "requests",
    "rustbpe",       # fast BPE tokenizer (Rust)
    "tiktoken",      # tokenizer utils
]
for pkg in core:
    ok = pip(pkg)
    print(f"  {'OK' if ok else 'FAIL':4s}  {pkg}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradient 2.0.6 requires attrs<=19, but you have attrs 23.1.0 which is incompatible.
gradient 2.0.6 requires PyYAML==5.*, but you have pyyaml 6.0.3 which is incompatible.
transformers 4.35.2 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.6.0 which is incompatible.
tokenizers 0.15.1 requires huggingface_hub<1.0,>=0.16.4, but you have huggingface-hub 1.6.0 which is incompatible.
datasets 2.14.5 requires huggingface-hub<1.0.0,>=0.14.0, but you have huggingface-hub 1.6.0 which is incompatible.


  OK    kernels


  OK    matplotlib


  OK    numpy


  OK    pandas


  OK    pyarrow


  OK    requests


  OK    rustbpe
  OK    tiktoken


## 3. A4000 Patches

### 3a. Flash Attention 3 check

FA3 targets H100 (sm_90). On A4000 (sm_86) it uses `kernels-community/flash-attn3`.
Test below â€” if it fails run **3b** to swap in PyTorch SDPA.

In [7]:
import torch
cap = torch.cuda.get_device_capability()
repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"
print(f"Testing: {repo}")
try:
    from kernels import get_kernel
    fa3 = get_kernel(repo).flash_attn_interface
    q = torch.randn(2, 16, 4, 32, dtype=torch.bfloat16, device="cuda")
    k, v = torch.randn_like(q), torch.randn_like(q)
    out = fa3.flash_attn_func(q, k, v, causal=True)
    print(f"FA3 OK: output shape {out.shape}")
except Exception as e:
    print(f"FA3 FAILED: {e}")
    print("-> Run cell 3b to patch with PyTorch SDPA fallback")

Testing: kernels-community/flash-attn3


Fetching 0 files: 0it [00:00, ?it/s]

FA3 FAILED: Cannot install kernel from repo kernels-community/flash-attn3 (revision: main)
-> Run cell 3b to patch with PyTorch SDPA fallback


In [8]:
# 3b. Patch train.py: swap FA3 for PyTorch SDPA  (only run if 3a failed)
with open("train.py", "r") as f:
    src = f.read()

if "# [A4000-PATCHED]" in src:
    print("Already patched.")
else:
    old = (
        "from kernels import get_kernel\n"
        "cap = torch.cuda.get_device_capability()\n"
        "# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs\n"
        'repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"\n'
        "fa3 = get_kernel(repo).flash_attn_interface"
    )
    new = (
        "# [A4000-PATCHED] PyTorch SDPA replaces FA3 (FA3 unreliable on Ampere)\n"
        "import torch.nn.functional as _Fsdpa\n"
        "class _FA3:\n"
        "    @staticmethod\n"
        "    def flash_attn_func(q, k, v, causal=True, window_size=(-1, 0)):\n"
        "        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)\n"
        "        ws = window_size[0] if isinstance(window_size, (tuple, list)) else window_size\n"
        "        if ws > 0 and ws < q.shape[2]:\n"
        "            T = q.shape[2]\n"
        "            m = torch.ones(T,T,dtype=torch.bool,device=q.device).tril()\n"
        "            m = m & torch.ones(T,T,dtype=torch.bool,device=q.device).triu(-(ws-1))\n"
        "            am = torch.zeros(T,T,dtype=q.dtype,device=q.device).masked_fill_(~m, float('-inf'))\n"
        "            out = _Fsdpa.scaled_dot_product_attention(q, k, v, attn_mask=am)\n"
        "        else:\n"
        "            out = _Fsdpa.scaled_dot_product_attention(q, k, v, is_causal=causal)\n"
        "        return out.transpose(1, 2)\n"
        "class _FA3Iface:\n"
        "    flash_attn_func = staticmethod(_FA3.flash_attn_func)\n"
        "fa3 = _FA3Iface()"
    )
    if old in src:
        src = src.replace(old, new)
        open("train.py", "w").write(src)
        print("Patched: FA3 -> PyTorch SDPA")
    else:
        print("WARNING: FA3 block not found â€” check train.py manually")

Patched: FA3 -> PyTorch SDPA


### 3c. Tune hyperparameters for 16 GB VRAM

| File | Parameter | H100 default | A4000 | Reason |
|------|-----------|-------------|-------|--------|
| `prepare.py` | `MAX_SEQ_LEN` | 2048 | **1024** | Halves activation memory |
| `prepare.py` | `EVAL_TOKENS` | 40x | **10x** | Faster eval |
| `train.py` | `DEVICE_BATCH_SIZE` | 128 | **16** | Primary VRAM fix |
| `train.py` | `TOTAL_BATCH_SIZE` | 2^19 | **2^17** | grad_accum = 131072/(16Ã—1024) = 8 âœ“ |
| `train.py` | `WINDOW_PATTERN` | SSSL | **L** | Skip banded attention overhead |
| `train.py` | `DEPTH` | 8 | **6** | Smaller model |

In [9]:
with open("prepare.py", "r") as f:
    src = f.read()
for old, new in [
    ("MAX_SEQ_LEN = 2048", "MAX_SEQ_LEN = 1024"),
    ("EVAL_TOKENS = 40 * 524288", "EVAL_TOKENS = 10 * 524288"),
]:
    if old in src: src = src.replace(old, new); print(f"prepare.py: {old!r} -> {new!r}")
    elif new in src: print(f"prepare.py: already set: {new!r}")
    else: print(f"WARNING: not found: {old!r}")
open("prepare.py", "w").write(src)
print("prepare.py done")

prepare.py: 'MAX_SEQ_LEN = 2048' -> 'MAX_SEQ_LEN = 1024'
prepare.py: 'EVAL_TOKENS = 40 * 524288' -> 'EVAL_TOKENS = 10 * 524288'
prepare.py done


In [10]:
with open("train.py", "r") as f:
    src = f.read()
for old, new in [
    ("TOTAL_BATCH_SIZE = 2**19", "TOTAL_BATCH_SIZE = 2**17"),
    ("DEVICE_BATCH_SIZE = 128",  "DEVICE_BATCH_SIZE = 16"),
    ('WINDOW_PATTERN = "SSSL"',  'WINDOW_PATTERN = "L"'),
    ("DEPTH = 8",                "DEPTH = 6"),
]:
    if old in src: src = src.replace(old, new); print(f"train.py: {old!r} -> {new!r}")
    elif new in src: print(f"train.py: already set: {new!r}")
    else: print(f"WARNING: not found: {old!r}")
open("train.py", "w").write(src)
import re
tbs = eval(re.search(r'TOTAL_BATCH_SIZE = (.+)', src).group(1))
dbs = int(re.search(r'DEVICE_BATCH_SIZE = (\d+)', src).group(1))
seq = int(re.search(r'MAX_SEQ_LEN = (\d+)', open('prepare.py').read()).group(1))
g = tbs // (dbs * seq)
ok = tbs % (dbs * seq) == 0
print(f'\nGrad accum: {tbs}/({dbs}*{seq}) = {g}  {"OK" if ok else "MISMATCH"}')

train.py: 'TOTAL_BATCH_SIZE = 2**19' -> 'TOTAL_BATCH_SIZE = 2**17'
train.py: 'DEVICE_BATCH_SIZE = 128' -> 'DEVICE_BATCH_SIZE = 16'
train.py: 'WINDOW_PATTERN = "SSSL"' -> 'WINDOW_PATTERN = "L"'
train.py: 'DEPTH = 8' -> 'DEPTH = 6'

Grad accum: 131072/(16*1024) = 8  OK


## 4. Embedding Research Libraries

Extra packages for researching resource-efficient embedding methods inside the autoresearch loop.

| Library | Purpose |
|---------|--------|
| `einops` | Clean tensor reshape/rearrange â€” essential for factorized & compositional embeddings |
| `bitsandbytes` | int8 / int4 quantized embeddings and linear layers (Ampere supported) |
| `vector-quantize-pytorch` | VQ-VAE / product quantization codebook embeddings |
| `triton` | Write custom CUDA kernels inline (ships with PyTorch, listed for clarity) |
| `scipy` | Cosine similarity, clustering, embedding analysis utils |
| `scikit-learn` | PCA, k-means â€” useful for embedding space analysis |
| `rich` | Pretty experiment tables in notebook output |
| `safetensors` | Fast tensor save/load for embedding checkpoints |

In [11]:
import sys, subprocess

research_libs = [
    # Embedding manipulation & factorization
    ("einops",                   "tensor reshape for factorized/compositional embeddings"),

    # Quantized embeddings
    ("bitsandbytes",             "int8/int4 quantized embeddings (Ampere sm_86 supported)"),

    # Codebook / product quantization
    ("vector-quantize-pytorch",  "VQ / residual-VQ / product-quantization codebooks"),

    # Custom CUDA kernels
    ("triton",                   "write inline GPU kernels for custom embedding ops"),

    # Analysis
    ("scipy",                    "cosine sim, clustering, embedding geometry analysis"),
    ("scikit-learn",             "PCA/k-means on embedding spaces"),

    # Utilities
    ("rich",                     "pretty tables for experiment results in notebook"),
    ("safetensors",              "fast save/load for embedding checkpoints"),
]

print(f"{'Library':<35} {'Status':<8} Purpose")
print("-" * 80)
for pkg, purpose in research_libs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True)
    status = "OK" if r.returncode == 0 else "FAIL"
    print(f"{pkg:<35} {status:<8} {purpose}")
print("\nAll done.")

Library                             Status   Purpose
--------------------------------------------------------------------------------
einops                              OK       tensor reshape for factorized/compositional embeddings
bitsandbytes                        OK       int8/int4 quantized embeddings (Ampere sm_86 supported)
vector-quantize-pytorch             OK       VQ / residual-VQ / product-quantization codebooks
triton                              OK       write inline GPU kernels for custom embedding ops
scipy                               OK       cosine sim, clustering, embedding geometry analysis
scikit-learn                        OK       PCA/k-means on embedding spaces
rich                                OK       pretty tables for experiment results in notebook
safetensors                         OK       fast save/load for embedding checkpoints

All done.


In [12]:
# Quick import check â€” verify all research libs are importable
import importlib, sys

checks = [
    ("einops",           "einops"),
    ("bitsandbytes",     "bitsandbytes"),
    ("vector_quantize_pytorch", "vector_quantize_pytorch"),
    ("triton",           "triton"),
    ("scipy",            "scipy"),
    ("sklearn",          "scikit-learn"),
    ("rich",             "rich"),
    ("safetensors",      "safetensors"),
    ("torch",            "torch"),
    ("tiktoken",         "tiktoken"),
]
for import_name, pkg_name in checks:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, '__version__', 'ok')
        print(f"  OK   {import_name:<30} {ver}")
    except ImportError as e:
        print(f"  FAIL {import_name:<30} {e}")

  OK   einops                         0.8.2
  FAIL bitsandbytes                   No module named 'triton.ops'
  OK   vector_quantize_pytorch        ok
  OK   triton                         3.6.0
  OK   scipy                          1.11.2
  OK   sklearn                        1.3.0
  OK   rich                           ok
  OK   safetensors                    0.4.0
  OK   torch                          2.1.1+cu121
  OK   tiktoken                       0.12.0


## 5. Prepare Data

One-time download (10 training shards + 1 pinned val shard) + BPE tokenizer training.
~2â€“5 min. Cached at `~/.cache/autoresearch/`.

In [13]:
import os, subprocess, sys
cache = os.path.expanduser("~/.cache/autoresearch")
tok = os.path.join(cache, "tokenizer", "tokenizer.pkl")
data = os.path.join(cache, "data")
if os.path.exists(tok):
    shards = len([f for f in os.listdir(data) if f.endswith(".parquet")]) if os.path.exists(data) else 0
    print(f"Already prepared: {shards} shards, tokenizer exists")
else:
    print("Running prepare.py --num-shards 10 ...")
    r = subprocess.run([sys.executable, "prepare.py", "--num-shards", "10"])
    print("Done!" if r.returncode == 0 else f"Error: returncode={r.returncode}")

Running prepare.py --num-shards 10 ...
Cache directory: /root/.cache/autoresearch

Data: downloading 11 shards (0 already exist)...
  Downloaded shard_00000.parquet
  Downloaded shard_00007.parquet
  Downloaded shard_00003.parquet
  Downloaded shard_00002.parquet
  Downloaded shard_00006.parquet
  Downloaded shard_06542.parquet
  Downloaded shard_00001.parquet
  Downloaded shard_00008.parquet
  Downloaded shard_00004.parquet
Data: 11/11 shards ready at /root/.cache/autoresearch/data

Tokenizer: training BPE tokenizer...
Tokenizer: trained in 42.4s, saved to /root/.cache/autoresearch/tokenizer/tokenizer.pkl
Tokenizer: building token_bytes lookup...
Tokenizer: saved token_bytes to /root/.cache/autoresearch/tokenizer/token_bytes.pt
Tokenizer: sanity check passed (vocab_size=8192)

Done! Ready to train.
Done!


## 6. Baseline Training Run

5-minute run on unmodified `train.py`. Sets the `val_bpb` baseline all experiments must beat.

In [14]:
import subprocess, sys, time
print("Starting baseline run (~5 min + startup)... output -> run.log")
t0 = time.time()
r = subprocess.run(
    [sys.executable, "train.py"],
    stdout=open("run.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Finished in {time.time()-t0:.0f}s  (returncode={r.returncode})")

Starting baseline run (~5 min + startup)... output -> run.log
Finished in 354s  (returncode=0)


In [15]:
import subprocess
r = subprocess.run(
    ["grep", "-E",
     "^val_bpb:|^peak_vram_mb:|^num_params_M:|^depth:|^total_tokens_M:|^mfu_percent:",
     "run.log"],
    capture_output=True, text=True
)
if r.stdout.strip():
    print("=== BASELINE RESULTS ===")
    print(r.stdout)
else:
    print("No results â€” run may have crashed. Last 40 lines of run.log:")
    print("".join(open("run.log").readlines()[-40:]))

=== BASELINE RESULTS ===
val_bpb:          1.418095
peak_vram_mb:     1869.3
mfu_percent:      2.88
total_tokens_M:   91.2
num_params_M:     13.4
depth:            6



In [16]:
# Full log tail (last 3000 chars)
print(open("run.log").read()[-3000:])

oss: 3.983217 | lrm: 0.06 | dt: 430ms | tok/sec: 304,861 | mfu: 2.9% | epoch: 1 | remaining: 9s    
step 00674 (96.9%) | loss: 3.983963 | lrm: 0.06 | dt: 437ms | tok/sec: 300,221 | mfu: 2.9% | epoch: 1 | remaining: 9s    
step 00675 (97.1%) | loss: 3.995265 | lrm: 0.06 | dt: 434ms | tok/sec: 301,885 | mfu: 2.9% | epoch: 1 | remaining: 8s    
step 00676 (97.2%) | loss: 3.997415 | lrm: 0.06 | dt: 438ms | tok/sec: 299,489 | mfu: 2.9% | epoch: 1 | remaining: 8s    
step 00677 (97.4%) | loss: 4.000908 | lrm: 0.05 | dt: 430ms | tok/sec: 304,906 | mfu: 2.9% | epoch: 1 | remaining: 7s    
step 00678 (97.5%) | loss: 3.984173 | lrm: 0.05 | dt: 431ms | tok/sec: 303,770 | mfu: 2.9% | epoch: 1 | remaining: 7s    
step 00679 (97.6%) | loss: 3.970885 | lrm: 0.05 | dt: 436ms | tok/sec: 300,878 | mfu: 2.9% | epoch: 1 | remaining: 7s    
step 00680 (97.8%) | loss: 3.971475 | lrm: 0.04 | dt: 439ms | tok/sec: 298,874 | mfu: 2.9% | epoch: 1 | remaining: 6s    
step 00681 (97.9%) | loss: 3.975681 | lrm: 0.0

## 7. Results & Experiment Runner

Handles `results.tsv` on the cloud kernel's local storage.

**Run the init cell first**, then use `run_experiment('description')` for every training run — it runs `train.py`, parses the output, auto-determines keep/discard vs current best, and logs to `results.tsv`.

In [23]:
# Initialize results.tsv (safe to re-run — won't overwrite existing rows)
import os

TSV = 'results.tsv'
HEADER = 'commit	val_bpb	memory_gb	status	description'

if not os.path.exists(TSV):
    with open(TSV, 'w') as f:
        f.write(HEADER + '\n')
    print(f'Created {TSV}')
else:
    rows = open(TSV).readlines()
    print(f'{TSV} exists — {len(rows)-1} experiment(s) logged')
    if len(rows) > 1:
        print(open(TSV).read())


results.tsv exists — 0 experiment(s) logged


In [24]:
import subprocess, sys, time, os, re

TSV = 'results.tsv'

def _run_id():
    r = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else time.strftime('%H%M%S')

def _parse_log(path):
    try:
        txt = open(path).read()
    except FileNotFoundError:
        return None
    m = {}
    for k in ['val_bpb', 'peak_vram_mb', 'num_params_M', 'mfu_percent', 'total_tokens_M']:
        hit = re.search(rf'^{k}:\s+([\d.]+)', txt, re.MULTILINE)
        if hit:
            m[k] = float(hit.group(1))
    return m if 'val_bpb' in m else None

def _log(run_id, val_bpb, memory_gb, status, description):
    if not os.path.exists(TSV):
        open(TSV, 'w').write('commit\tval_bpb\tmemory_gb\tstatus\tdescription\n')
    with open(TSV, 'a') as f:
        f.write(f'{run_id}\t{val_bpb:.6f}\t{memory_gb:.1f}\t{status}\t{description}\n')
    print(f'Logged -> {run_id} | bpb={val_bpb:.6f} | {memory_gb:.1f}GB | {status} | {description}')

def _best_bpb():
    if not os.path.exists(TSV):
        return float('inf')
    rows = open(TSV).readlines()[1:]
    kept = [float(r.split('\t')[1]) for r in rows
            if r.strip() and len(r.split('\t')) >= 4 and r.split('\t')[3] == 'keep']
    return min(kept) if kept else float('inf')

def run_experiment(description, log_file='run.log', status=None):
    """
    Run train.py, parse results, auto-log to results.tsv.

    Args:
        description : label for this experiment, e.g. 'baseline', 'EMBED_RANK=128'
        log_file    : output log path (default: run.log)
        status      : 'keep'/'discard' override — if None, auto-compares to current best
    Returns:
        (val_bpb, memory_gb) or None on crash
    """
    print(f'Starting: {description} -> {log_file}')
    print('(~5 min + startup)')
    t0 = time.time()
    subprocess.run(
        [sys.executable, 'train.py'],
        stdout=open(log_file, 'w'),
        stderr=subprocess.STDOUT,
    )
    elapsed = time.time() - t0
    m = _parse_log(log_file)
    rid = _run_id()

    if m is None:
        print(f'CRASH after {elapsed:.0f}s. Last 30 lines:')
        print(''.join(open(log_file).readlines()[-30:]))
        _log(rid, 0.0, 0.0, 'crash', description)
        return None

    bpb   = m['val_bpb']
    vram  = m.get('peak_vram_mb', 0) / 1024
    best  = _best_bpb()
    s     = status if status else ('keep' if bpb < best else 'discard')

    _log(rid, bpb, vram, s, description)

    print(f'\nval_bpb   = {bpb:.6f}  (best so far: {best:.6f})')
    print(f'peak_vram = {vram:.1f} GB')
    print(f'params    = {m.get("num_params_M", 0):.1f}M')
    print(f'mfu       = {m.get("mfu_percent", 0):.1f}%')
    print(f'tokens    = {m.get("total_tokens_M", 0):.0f}M')
    print(f'elapsed   = {elapsed:.0f}s')
    print(f'status    = {s}')
    return bpb, vram

print('run_experiment() ready.')
print('  run_experiment("baseline")           # first run')
print('  run_experiment("EMBED_RANK=128")      # after editing train.py')
print('  run_experiment("my-idea", status="keep")  # force keep')


run_experiment() ready.
  run_experiment("baseline")           # first run
  run_experiment("EMBED_RANK=128")      # after editing train.py
  run_experiment("my-idea", status="keep")  # force keep


In [22]:
# View results.tsv as a formatted table
import os

TSV = 'results.tsv'
if not os.path.exists(TSV):
    print('results.tsv not found — run init cell first.')
else:
    rows = open(TSV).readlines()
    if len(rows) <= 1:
        print('No experiments logged yet.')
    else:
        kept = [float(r.split('\t')[1]) for r in rows[1:]
                if r.strip() and len(r.split('\t')) >= 4 and r.split('\t')[3] == 'keep']
        best_bpb = min(kept) if kept else None
        icon = {'keep': 'OK ', 'discard': '-- ', 'crash': 'ERR'}
        print(f"{'COMMIT':<10} {'VAL_BPB':<12} {'VRAM_GB':<9} {'ST':<4} DESCRIPTION")
        print('-' * 68)
        for row in rows[1:]:
            if not row.strip(): continue
            p = row.strip().split('\t')
            if len(p) < 5: continue
            commit, bpb, mem, st, desc = p[0], p[1], p[2], p[3], p[4]
            flag = ' <-- best' if best_bpb and abs(float(bpb) - best_bpb) < 1e-8 and st == 'keep' else ''
            print(f"{commit:<10} {bpb:<12} {mem:<9} {icon.get(st,'?  '):<4} {desc}{flag}")
        print(f'\n{len(rows)-1} experiments total')


No experiments logged yet.


## 8. Autonomous Experiment Loop

Run **cell 25** once to load the helpers, then **cell 26** to execute the full Tier-1 sweep autonomously.

Each call to `auto_experiment()` patches `train.py`, commits, trains for 5 min, logs the result, and **reverts the commit** if there's no improvement.

In [ ]:
import subprocess, re, os

def patch_train_py(replacements):
    """Apply {old: new} string replacements to train.py."""
    txt = open('train.py').read()
    for old, new in replacements.items():
        if old not in txt:
            raise ValueError(f'Pattern not found in train.py: {repr(old[:80])}')
        txt = txt.replace(old, new, 1)
    open('train.py', 'w').write(txt)
    print(f'train.py patched ({len(replacements)} change(s))')

def get_embed_rank():
    """Read current EMBED_RANK value from train.py."""
    m = re.search(r'^EMBED_RANK\s*=\s*(\d+)', open('train.py').read(), re.MULTILINE)
    return int(m.group(1)) if m else None

def _commit_train_py(message):
    subprocess.run(['git', 'add', 'train.py'], check=True)
    r = subprocess.run(['git', 'commit', '-m', message], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    return r.returncode == 0

def _git_reset(n=1):
    subprocess.run(['git', 'reset', '--hard', f'HEAD~{n}'], check=True)
    print(f'Reverted {n} commit(s) -- train.py restored.')

def auto_experiment(description, replacements, log_file='run.log'):
    """
    Full cycle: patch train.py -> commit -> train 5 min -> log -> keep or revert.
    Returns (val_bpb, memory_gb) or None on crash.
    """
    print('
' + '='*62)
    print(f'EXPERIMENT: {description}')
    print('='*62)
    best_before = _best_bpb()
    try:
        patch_train_py(replacements)
    except ValueError as e:
        print(f'SKIP -- {e}')
        return None
    _commit_train_py(f'experiment: {description}')
    result = run_experiment(description, log_file=log_file)
    if result and result[0] < best_before:
        print(f'KEPT -- new best {result[0]:.6f} (was {best_before:.6f})')
    else:
        verdict = 'CRASH' if result is None else f'no improvement ({result[0]:.6f} >= {best_before:.6f})'
        print(f'DISCARD -- {verdict} -- reverting commit')
        _git_reset(1)
    return result

print('auto_experiment() ready.')


In [ ]:
# Tier 1: log baseline if needed, then sweep EMBED_RANK
import os

TSV = 'results.tsv'
rows = open(TSV).readlines() if os.path.exists(TSV) else ['header']
if len(rows) <= 1:
    print('No baseline logged -- parsing last run.log as baseline...')
    m = _parse_log('run.log')
    if m:
        rid = _run_id()
        _log(rid, m['val_bpb'], m.get('peak_vram_mb', 0)/1024, 'keep', 'baseline-r64')
        print(f'Baseline logged: val_bpb={m["val_bpb"]:.6f}')
    else:
        print('run.log not found -- running baseline now...')
        run_experiment('baseline-r64')

for target_rank in [128, 32, 96]:
    current_rank = get_embed_rank()
    if current_rank == target_rank:
        print(f'Already at EMBED_RANK={target_rank}, skipping.')
        continue
    auto_experiment(
        f'EMBED_RANK={target_rank}',
        {f'EMBED_RANK = {current_rank}': f'EMBED_RANK = {target_rank}'}
    )

print('\n' + '='*62)
print('Tier 1 complete. Best rank so far:', get_embed_rank())
print('Run cell 23 for the full results table.')
